In [ ]:
from google.colab import files
uploaded = files.upload()


Saving script1.sql to script1.sql


In [ ]:
uploaded = files.upload()

Saving 1343 script2.sql to 1343 script2.sql


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('pizzapizza.sqlite')
cursor = conn.cursor()

with open('script1.sql', 'r', encoding='utf-8') as f1:
    cursor.executescript(f1.read())
conn.commit()

Report 1: Inventory Reorder Alert

In [ ]:
query1 = """
SELECT
    fl.store_id,
    fl.street_address,
    rm.name AS material_name,
    rm.quantity_in_stock,
    rm.reorder_level,
    (rm.reorder_level - rm.quantity_in_stock) AS suggested_order_quantity
FROM RawMaterial rm
JOIN FranchiseLocation fl ON rm.store_id = fl.store_id
WHERE rm.quantity_in_stock <= rm.reorder_level
ORDER BY fl.store_id, suggested_order_quantity DESC;
"""
df1 = pd.read_sql_query(query1, conn)
display(df1)


,store_id,street_address,material_name,quantity_in_stock,reorder_level,suggested_order_quantity


Report 2: Menu Item Revenue & Customization Analysis

In [ ]:
query2 = """
    SELECT
        mi.name AS menu_item,
        mi.category,
        SUM(oi.quantity) AS total_items_sold,
        SUM(oi.quantity * oi.unit_price) AS base_revenue,
        IFNULL(SUM(pc.extra_charge), 0) AS customization_revenue,
        (SUM(oi.quantity * oi.unit_price) + IFNULL(SUM(pc.extra_charge), 0)) AS total_revenue
    FROM MenuItem mi
    JOIN OrderItem oi ON mi.item_id = oi.item_id
    LEFT JOIN PizzaCustomization pc ON oi.order_item_id = pc.order_item_id
    GROUP BY mi.item_id, mi.name, mi.category
    ORDER BY total_revenue DESC;
"""
df2 = pd.read_sql_query(query2, conn)
display(df2)

,menu_item,category,total_items_sold,base_revenue,customization_revenue,total_revenue
0,Pepperoni Pizza,Pizza,3,48.97,1.50,50.47
1,BBQ Chicken Pizza,Pizza,2,36.48,1.00,37.48
2,Hawaiian Pizza,Pizza,2,34.98,1.75,36.73
3,Margherita Pizza,Pizza,2,33.98,1.25,35.23
4,Veggie Pizza,Pizza,2,32.48,1.25,33.73
5,Garlic Bread,Side,3,17.97,0.00,17.97
6,Cola,Drink,6,17.94,0.00,17.94
7,Chicken Wings,Side,1,9.99,0.00,9.99
8,Caesar Salad,Side,1,7.49,0.00,7.49
9,Bottled Water,Drink,3,5.97,0.00,5.97


Report 3: Driver Performance and Tip Analysis

In [ ]:
query3 = """
    SELECT
        e.name AS driver_name,
        d.vehicle_type,
        COUNT(del.order_id) AS total_deliveries,
        SUM(del.tip_amount) AS total_tips_collected,
        ROUND(AVG(del.tip_amount), 2) AS average_tip_per_delivery
    FROM Driver d
    JOIN Employee e ON d.employee_id = e.employee_id
    JOIN Delivery del ON d.employee_id = del.driver_id
    GROUP BY d.employee_id, e.name, d.vehicle_type
    ORDER BY total_deliveries DESC, total_tips_collected DESC;
"""
df3 = pd.read_sql_query(query3, conn)
display(df3)

,driver_name,vehicle_type,total_deliveries,total_tips_collected,average_tip_per_delivery
0,Hugo Ma,Car,1,5.00,5.00
1,Leo Tang,Bike,1,4.50,4.50
2,Felix Zhou,Car,1,4.00,4.00
3,Noah Xie,Bike,1,3.80,3.80
4,George Lin,Bike,1,3.50,3.50
5,Kevin He,Car,1,3.25,3.25
6,Daniel Park,Car,1,3.00,3.00
7,Mason Qiu,Car,1,2.75,2.75
8,Ethan Kim,Bike,1,2.50,2.50
9,Ian Luo,Bike,1,2.00,2.00


Report 4: Customer Loyalty & Engagement

In [ ]:
query4 = """
    SELECT
        c.member_id,
        c.name AS customer_name,
        c.points_balance,
        COUNT(co.order_id) AS total_orders_placed
    FROM Customer c
    JOIN CustomerOrder co ON c.member_id = co.member_id
    GROUP BY c.member_id, c.name, c.points_balance
    ORDER BY c.points_balance DESC, total_orders_placed DESC
    LIMIT 10;
"""
df4 = pd.read_sql_query(query4, conn)
display(df4)

,member_id,customer_name,points_balance,total_orders_placed
0,C003,Cathy Wang,200,2
1,C005,Emma Liu,150,2
2,C009,Ivy Zhao,140,1
3,C001,Alice Chen,120,1
4,C008,Henry Xu,95,1
5,C007,Grace Wu,80,1
6,C004,David Zhang,60,1
7,C002,Brian Li,45,1
8,C006,Frank Sun,35,1
9,C010,Jason Gao,20,1


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']

# (Report 1 - Inventory Reorder Alert)
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 5))

sns.barplot(data=df1, x='material_name', y='suggested_order_quantity', palette='Reds_r')
plt.title('Inventory Reorder Alert: Suggested Order Quantities', fontsize=16, fontweight='bold')
plt.xlabel('Raw Material (Out of Stock)', fontsize=12)
plt.ylabel('Suggested Order Quantity', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# (Report 2 - Revenue & Customization)
# ------------------------------------------------------------------------------
df2_plot = df2.set_index('menu_item')[['base_revenue', 'customization_revenue']]

ax = df2_plot.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#4C72B0', '#55A868'])
plt.title('Revenue Breakdown: Base Price vs. Customizations', fontsize=16, fontweight='bold')
plt.xlabel('Menu Item', fontsize=12)
plt.ylabel('Total Revenue (CAD)', fontsize=12)
plt.legend(['Base Revenue', 'Customization Revenue (Extra)'], title='Revenue Source')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# (Report 3 - Driver Performance)
# ------------------------------------------------------------------------------
styled_df3 = df3.style.highlight_max(subset=['total_deliveries', 'total_tips_collected', 'average_tip_per_delivery'],
                                     color='lightgreen', axis=0)
display(styled_df3)


# (Report 4 - Customer Loyalty & Engagement)
# ------------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

sns.scatterplot(data=df4, x='total_orders_placed', y='points_balance',
                s=200, color='#C44E52', edgecolor='black', alpha=0.8)

for i in range(df4.shape[0]):
    plt.text(df4['total_orders_placed'][i] + 0.1,
             df4['points_balance'][i] + 0.5,
             df4['customer_name'][i],
             horizontalalignment='left', size='medium', color='black', weight='semibold')

plt.title('VIP Customer Matrix: Orders vs. Points Balance', fontsize=16, fontweight='bold')
plt.xlabel('Total Orders Placed', fontsize=12)
plt.ylabel('Points Balance', fontsize=12)
plt.axvline(x=df4['total_orders_placed'].mean(), color='grey', linestyle='--', label='Average Orders')
plt.axhline(y=df4['points_balance'].mean(), color='blue', linestyle='--', label='Average Points')
plt.legend()
plt.tight_layout()
plt.show()